# RAG fundamentals

**Track:** RAG. **Level:** Intermediate. **CPU:** yes. **Keys:** no.

## Goals

- Split a short document into sentences and into fixed character windows.
- Retrieve with TF-IDF over a numpy store.
- Answer by quoting a chunk and citing its id, or abstain when the query shares no terms.

## Sources and license

Original notebook. The outline follows the retrieval-augmented generation setup in Lewis et al. (2020) and the introductory RAG labs of the Full Stack LLM bootcamp. Those labs are not copied. TF-IDF is the scikit-learn vectorizer (BSD-3), implemented here directly so the weighting is visible. The portfolio RAG lab is the production-shaped version of this pipeline. This one is the smaller mechanism.

## Concepts

A chunk is the unit a citation can point at. A fixed window can cut a fact in half. A sentence window usually does not. Retrieval here is lexical: a query with no shared tokens scores zero, and zero is the abstention signal. That is the wrong behavior for a paraphrase and the right behavior for a question the corpus does not contain. The answer function is extractive. It does not call a language model. A generative reader belongs on top of a retriever you have already measured.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import math
import re
from collections import Counter

import numpy as np

from labs._shared.seed import seed_everything

seed_everything(0)
TOKEN = re.compile(r"[a-z0-9]+")

def tokenize(text):
    return TOKEN.findall(text.lower())


## Guided exercise: chunk, embed, retrieve, abstain

The document is original. One fact, the price, sits in a single sentence. A 24-character window is short enough to separate the product name from the price. The sentence chunk is not.


In [ ]:
DOCUMENT = (
    "The storage service is called lighthouse. "
    "Its monthly price is 12 dollars per seat. "
    "Backups run every night and are kept for 30 days."
)

def fixed_windows(text, size):
    return [text[i : i + size] for i in range(0, len(text), size)]

def sentences(text):
    return [part.strip() for part in text.split(". ") if part.strip()]

windows = fixed_windows(DOCUMENT, 24)
sents = sentences(DOCUMENT)
print("windows", windows)
print("sentences", sents)
assert any("lighthouse" in chunk and "12" not in chunk for chunk in windows)
assert any("lighthouse" in chunk and "12" in chunk for chunk in [" ".join(sents)])

class TfidfIndex:
    def fit(self, ids, texts):
        self.ids = list(ids)
        self.texts = list(texts)
        docs = [tokenize(text) for text in texts]
        df = Counter()
        for doc in docs:
            for term in set(doc):
                df[term] += 1
        self.idf = {term: math.log((1 + len(docs)) / (1 + df[term])) + 1.0 for term in df}
        self.vocab = {term: i for i, term in enumerate(sorted(self.idf))}
        self.matrix = self.embed(texts)

    def embed(self, texts):
        matrix = np.zeros((len(texts), max(len(self.vocab), 1)))
        for row, text in enumerate(texts):
            for term, count in Counter(tokenize(text)).items():
                col = self.vocab.get(term)
                if col is not None:
                    matrix[row, col] = count * self.idf[term]
            norm = np.linalg.norm(matrix[row])
            if norm:
                matrix[row] /= norm
        return matrix

    def rank(self, query, k=3, minimum=0.05):
        vector = self.embed([query])[0]
        if np.linalg.norm(vector) == 0:
            return []
        scores = self.matrix @ vector
        order = np.argsort(-scores)
        hits = []
        for i in order:
            if scores[i] < minimum:
                break
            hits.append((self.ids[int(i)], float(scores[i]), self.texts[int(i)]))
            if len(hits) == k:
                break
        return hits

index = TfidfIndex()
index.fit([f"s{i}" for i in range(len(sents))], sents)
price = index.rank("monthly price per seat")
unknown = index.rank("zebrafish genome assembly quality")
print("price", price)
print("unknown", unknown)
assert price and "12" in price[0][2]
assert unknown == []

def answer(query):
    hits = index.rank(query)
    if not hits:
        return "I do not know."
    doc_id, _score, text = hits[0]
    return f"{text.strip('.')} [{doc_id}]"

print(answer("monthly price per seat"))
print(answer("zebrafish genome assembly quality"))
assert "[s" in answer("monthly price per seat")
assert "12" in answer("monthly price per seat")
assert answer("zebrafish genome assembly quality") == "I do not know."
print("LAB OK")


## Your turn

Retrieve `nightly backups`. The hit should be the backups sentence, not the price sentence.

## Solution


In [ ]:
hits = index.rank("nightly backups")
print(hits)
assert hits and "30" in hits[0][2]


## Beyond the course

Build the index on the fixed windows instead of the sentences and ask for the price using the word `lighthouse`. Report whether any returned window contains both the name and the price. The sentence index did. This is the chunking ablation in one query.


In [ ]:
window_index = TfidfIndex()
window_index.fit([f"w{i}" for i in range(len(windows))], windows)
window_hits = window_index.rank("lighthouse price", k=4, minimum=0.0)
print(window_hits)
both = [hit for hit in window_hits if "lighthouse" in hit[2] and "12" in hit[2]]
print("windows containing both facts", len(both))
assert both == []


## Research questions

1. Abstention used a shared-token threshold. What relevant question does that rule refuse?
2. The quoted answer can be the wrong sentence with a legal citation. Which check from the portfolio RAG lab would catch it?
3. TF-IDF was refit on three sentences. What changes when the idf statistics come from a million documents that do not include this product?

## Further reading

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* (2020).
- scikit-learn TF-IDF user guide (BSD-3).
- Full Stack LLM bootcamp, retrieval section. Not copied.
- The portfolio lab `labs/portfolio/01_production_rag` adds hybrid search, stores, and a citation check.
